In [ ]:
# Normal:     0928
# IP, DeviceId, UserAgent
# 111.90.238.224, 435BB410-6433-453A-95C0-590E78543A64 , openrice_ios/7.21.0 (iphone; ios 18.5)
# 112.198.104.210, 72dd0c61450f9efd, openrice_android/7.21.0 (android; 35) [cph2799]
# 136.158.43.30, 8503e3f7ea531cf4, openrice_android/7.21.0 (android; 36) [sm-a566b]

# Abnormal (from Pakistan):
# 223.123.7.182,  cc21b7620eaf4e65, openrice_android/7.20.2 (android; 31)
# 223.123.7.182,  c9a76abc2baaf940,  openrice_android/7.20.2 (android; 31)
# 119.155.211.76, 907f26030cdd51e1, openrice_android/7.20.4 (android; 33) [infinix x6836]

In [1]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [3]:
sql = """
WITH target_users AS (
  SELECT
    'normal' AS user_type,
    '111.90.238.224' AS ip,
    '435bb410-6433-453a-95c0-590e78543a64' AS device_id,
    'openrice_ios/7.21.0 (iphone; ios 18.5)' AS user_agent

  UNION ALL SELECT
    'normal',
    '112.198.104.210',
    '72dd0c61450f9efd',
    'openrice_android/7.21.0 (android; 35) [cph2799]'

  UNION ALL SELECT
    'normal',
    '136.158.43.30',
    '8503e3f7ea531cf4',
    'openrice_android/7.21.0 (android; 36) [sm-a566b]'

  UNION ALL SELECT
    'abnormal',
    '223.123.7.182',
    'cc21b7620eaf4e65',
    'openrice_android/7.20.2 (android; 31)'

  UNION ALL SELECT
    'abnormal',
    '223.123.7.182',
    'c9a76abc2baaf940',
    'openrice_android/7.20.2 (android; 31)'

  UNION ALL SELECT
    'abnormal',
    '119.155.211.76',
    '907f26030cdd51e1',
    'openrice_android/7.20.4 (android; 33) [infinix x6836]'
)

SELECT
  target.user_type,
  pv.IP,
  pv.DeviceId,
  pv.SessionId,
  pv.Time,
  pv.EventAction,
  pv.EventLabel,
  pv.EventLabelRaw,
  pv.UserAgent
FROM `openrice-production.ORGA.PV_20260928` AS pv
INNER JOIN target_users AS target
  ON TRIM(CAST(pv.IP AS STRING)) = target.ip
  AND LOWER(pv.DeviceId) = target.device_id
  AND STARTS_WITH(LOWER(pv.UserAgent),target.user_agent)
  -- AND LOWER(pv.UserAgent) like '%' || target.user_agent || '%'
ORDER BY
  target.user_type,
  pv.IP,
  pv.DeviceId,
  pv.SessionId,
  pv.Time;
"""

In [4]:
df_bq = client.query(sql).result().to_dataframe()
df_bq

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,user_type,IP,DeviceId,SessionId,Time,EventAction,EventLabel,EventLabelRaw,UserAgent
0,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,320111,2026-09-28 01:17:58.300000+00:00,or.app.resume,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.21.0,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US
1,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,320111,2026-09-28 01:18:09.600000+00:00,or.lang.toggle,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:en;Ver:7.21.0,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US
2,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,320111,2026-09-28 01:18:14.400000+00:00,or.changelocation,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:en;Ver:7.21.0;RegionID:200;sr:Index,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US
3,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,320111,2026-09-28 01:18:18.700000+00:00,or.app.resume,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:200;Lang:en;Ver:7.21.0,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US
4,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,320111,2026-09-28 01:18:57.500000+00:00,user.login.google,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:200;Lang:en;Ver:7.21.0;sr:login,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US
5,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,320111,2026-09-28 01:19:00.100000+00:00,or.app.resume,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:200;Lang:en;Ver:7.21.0,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US
6,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,320111,2026-09-28 01:19:11.600000+00:00,or.app.resume,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:200;Lang:en;Ver:7.21.0,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US
7,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,320111,2026-09-28 01:19:44.700000+00:00,user.login.facebook,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:200;Lang:en;Ver:7.21.0;sr:login,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US
8,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,320111,2026-09-28 01:19:47.200000+00:00,or.app.resume,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:200;Lang:en;Ver:7.21.0,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US
9,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,320111,2026-09-28 01:20:02.800000+00:00,user.signup.info,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:200;Lang:en;Ver:7.21.0;sr:login,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US


In [5]:
sql_1 = """
WITH target_users AS (
  SELECT
    'normal' AS user_type,
    '111.90.238.224' AS ip,
    '435bb410-6433-453a-95c0-590e78543a64' AS device_id,
    'openrice_ios/7.21.0 (iphone; ios 18.5)' AS user_agent

  UNION ALL SELECT
    'normal',
    '112.198.104.210',
    '72dd0c61450f9efd',
    'openrice_android/7.21.0 (android; 35) [cph2799]'

  UNION ALL SELECT
    'normal',
    '136.158.43.30',
    '8503e3f7ea531cf4',
    'openrice_android/7.21.0 (android; 36) [sm-a566b]'

  UNION ALL SELECT
    'abnormal',
    '223.123.7.182',
    'cc21b7620eaf4e65',
    'openrice_android/7.20.2 (android; 31)'

  UNION ALL SELECT
    'abnormal',
    '223.123.7.182',
    'c9a76abc2baaf940',
    'openrice_android/7.20.2 (android; 31)'

  UNION ALL SELECT
    'abnormal',
    '119.155.211.76',
    '907f26030cdd51e1',
    'openrice_android/7.20.4 (android; 33) [infinix x6836]'
)

SELECT
  target.user_type,
  target.ip AS expected_ip,
  target.device_id AS expected_device_id,
  target.user_agent AS expected_user_agent,
  pv.IP AS observed_ip,
  pv.DeviceId AS observed_device_id,
  pv.UserAgent AS observed_user_agent,
  COUNT(*) AS event_count,
  MIN(pv.Time) AS first_event_time,
  MAX(pv.Time) AS last_event_time
FROM target_users AS target
LEFT JOIN `openrice-production.ORGA.PV_20260928` AS pv
  ON LOWER(TRIM(pv.DeviceId)) = target.device_id
GROUP BY
  target.user_type,
  target.ip,
  target.device_id,
  target.user_agent,
  pv.IP,
  pv.DeviceId,
  pv.UserAgent
ORDER BY target.user_type, target.device_id, event_count DESC;
"""

In [6]:
df_bq_1 = client.query(sql_1).result().to_dataframe()
df_bq_1

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,user_type,expected_ip,expected_device_id,expected_user_agent,observed_ip,observed_device_id,observed_user_agent,event_count,first_event_time,last_event_time
0,abnormal,119.155.211.76,907f26030cdd51e1,openrice_android/7.20.4 (android; 33) [infinix...,NaN,NaN,NaN,1,NaT,NaT
1,abnormal,223.123.7.182,c9a76abc2baaf940,openrice_android/7.20.2 (android; 31),NaN,NaN,NaN,1,NaT,NaT
2,abnormal,223.123.7.182,cc21b7620eaf4e65,openrice_android/7.20.2 (android; 31),NaN,NaN,NaN,1,NaT,NaT
3,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,openrice_ios/7.21.0 (iphone; ios 18.5),112.205.36.11,435bb410-6433-453a-95c0-590e78543a64,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US,128,2026-09-28 01:24:21+00:00,2026-09-28 21:38:58.100000+00:00
4,normal,111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,openrice_ios/7.21.0 (iphone; ios 18.5),111.90.238.224,435bb410-6433-453a-95c0-590e78543a64,OpenRice_iOS/7.21.0 (iPhone; iOS 18.5);Lang:en-US,15,2026-09-28 01:17:58.300000+00:00,2026-09-28 01:23:30+00:00
5,normal,112.198.104.210,72dd0c61450f9efd,openrice_android/7.21.0 (android; 35) [cph2799],NaN,NaN,NaN,1,NaT,NaT
6,normal,136.158.43.30,8503e3f7ea531cf4,openrice_android/7.21.0 (android; 36) [sm-a566b],NaN,NaN,NaN,1,NaT,NaT
